# Spectrogram denoising CNN

The network trains on magnitudes in `dataset/spectrograms` only. Wavs in `dataset/audio` are paired by `pair_id` and are used to rebuild phase for PESQ, STOI, and listening.

Select the **Python (.venv)** kernel before running. That interpreter is `.venv/bin/python`.

In [ ]:
import csv
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src" / "cnn" / "train.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "cnn" / "train.py").exists():
    raise SystemExit(f"Could not find src/cnn/train.py from {Path.cwd()}")

venv = (ROOT / ".venv" / "bin" / "python").resolve()
if Path(sys.executable).resolve() != venv:
    raise SystemExit(
        f"This notebook is running {sys.executable}.\n"
        f"Select the Python (.venv) kernel so it uses {venv}."
    )

sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "src" / "cnn"))
from train import TrainConfig, denoise_clip, evaluate_checkpoint, identity_check, load_split, train

SPEC_ROOT = ROOT / "dataset" / "spectrograms"
AUDIO_ROOT = ROOT / "dataset" / "audio"
print("kernel", sys.executable)

## One training pair

Input to the network is `log1p` of the noisy magnitude. The clean magnitude is the training target. Both stay linear on disk.

In [ ]:
noisy, clean, ids = load_split("train", SPEC_ROOT, AUDIO_ROOT)
print(ids[0], noisy.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, mag, title in zip(axes, (noisy[0], clean[0]), ("noisy", "clean")):
    image = ax.imshow(np.log1p(mag), origin="lower", aspect="auto", cmap="magma")
    ax.set_title(f"{ids[0]} {title}")
    ax.set_xlabel("frame")
    fig.colorbar(image, ax=ax, fraction=0.046)
axes[0].set_ylabel("frequency bin")
fig.tight_layout()

## Train

Change **one** field per run and give that run its own `run_name`. Compare learning rate `1e-3` vs `3e-4`, `base_channels` 16 vs 32, `target` `"mask"` vs `"map"`, and `dropout` 0 vs 0.2.

The identity check rebuilds a few validation clips from the stored noisy magnitude and the wav's own phase. Those scores have to match the noisy wav. Training then minimizes log-MSE on spectrograms. PESQ and STOI are computed from rebuilt audio on the validation split only.

In [ ]:
cfg = TrainConfig(
    run_name="mask_lr1e-3",
    lr=1e-3,
    batch_size=8,
    base_channels=16,
    target="mask",
    dropout=0.0,
    epochs=40,
    patience=8,
    seed=42,
)
identity_check(SPEC_ROOT, AUDIO_ROOT)
run_dir = train(cfg)
run_dir

## Is it overfitting or underfitting?

Solid line is train loss, dashed line is validation loss. The vertical line is the epoch with the best validation PESQ, which is the checkpoint in `best.pt`.

- A widening gap, with train loss still falling while validation loss flattens or rises, is overfitting. Stop at the marked epoch, or rerun with `dropout=0.2` or `base_channels=16`.
- Both curves stuck together at a high loss, and validation PESQ sitting on the noisy baseline printed during training, is underfitting. Train longer, try `base_channels=32`, or lower the learning rate if the loss is bouncing.

In [ ]:
def read_metrics(path):
    with open(path, newline="") as fh:
        rows = list(csv.DictReader(fh))
    epochs = [int(row["epoch"]) for row in rows]
    train_loss = [float(row["train_loss"]) for row in rows]
    val_loss = [float(row["val_loss"]) for row in rows]
    pesq = [float(row["val_pesq"]) if row["val_pesq"] else None for row in rows]
    return epochs, train_loss, val_loss, pesq

fig, ax = plt.subplots(figsize=(8, 4))
best_epoch = None
for metrics_path in sorted((ROOT / "runs").glob("*/metrics.csv")):
    epochs, train_loss, val_loss, pesq = read_metrics(metrics_path)
    name = metrics_path.parent.name
    ax.plot(epochs, train_loss, label=f"{name} train")
    ax.plot(epochs, val_loss, linestyle="--", label=f"{name} val")
    if metrics_path.parent == Path(run_dir):
        scored = [(epoch, score) for epoch, score in zip(epochs, pesq) if score is not None]
        if scored:
            best_epoch = max(scored, key=lambda item: item[1])[0]
if best_epoch is not None:
    ax.axvline(best_epoch, color="black", linewidth=1, label=f"best val PESQ (epoch {best_epoch})")
ax.set_xlabel("epoch")
ax.set_ylabel("log-MSE")
ax.set_title("Train vs validation loss")
ax.legend(fontsize=8)
fig.tight_layout()

## Listen to one validation clip

The denoised wav uses the network's magnitude and the noisy file's phase.

In [ ]:
clip = denoise_clip(Path(run_dir) / "best.pt", "val", index=0)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, mag, title in zip(
    axes,
    (clip["noisy_mag"], clip["pred_mag"], clip["clean_mag"]),
    ("noisy", "predicted", "clean"),
):
    image = ax.imshow(np.log1p(np.maximum(mag, 0)), origin="lower", aspect="auto", cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("frame")
fig.colorbar(image, ax=axes.ravel().tolist(), fraction=0.02)
axes[0].set_ylabel("frequency bin")
fig.suptitle(clip["pair_id"])
fig.tight_layout()

display(Markdown(f"**{clip['pair_id']}**"))
display(Markdown("Noisy"))
display(Audio(clip["noisy_wav"], rate=clip["sr"]))
display(Markdown("Denoised"))
display(Audio(clip["enhanced_wav"], rate=clip["sr"]))
display(Markdown("Clean"))
display(Audio(clip["clean_wav"], rate=clip["sr"]))

## Test split

Leave this off while you are comparing runs. Set `RUN_TEST = True` once, after validation PESQ has picked the checkpoint.

In [ ]:
RUN_TEST = False
CHECKPOINT = ROOT / "runs" / "mask_lr1e-3" / "best.pt"

if RUN_TEST:
    evaluate_checkpoint(CHECKPOINT, split="test")
else:
    print("Test split left sealed. Set RUN_TEST = True after choosing a checkpoint.")

## Results table

The comparison script trains each setting, scores the test split, and writes `runs/summary.csv`. The `selected` row is the highest validation PESQ. Run it from the repo root with `.venv/bin/python src/cnn/experiments.py`, then run this cell.

In [11]:
import sys
from pathlib import Path

repo = ROOT if "ROOT" in dir() else Path.cwd()
if not (repo / "src" / "cnn" / "experiments.py").exists():
    repo = repo.parent
sys.path.insert(0, str(repo / "src" / "cnn"))
from experiments import print_summary

print_summary(repo / "runs" / "summary.csv")

run           lr      channels  target  dropout  best_epoch  train_loss  val_loss  loss_gap  val_pesq  val_stoi  test_pesq  test_stoi  selected
------------  ------  --------  ------  -------  ----------  ----------  --------  --------  --------  --------  ---------  ---------  --------
noisy                                                                                        1.626     0.760     1.666      0.744              
mask_lr1e-3   0.001   16        mask    0        25          0.00392     0.00479   0.00087   2.456     0.790     2.209      0.748      no      
mask_lr3e-4   0.0003  16        mask    0        20          0.00402     0.00542   0.00141   2.348     0.795     2.163      0.755      no      
mask_ch32     0.001   32        mask    0        27          0.00371     0.00442   0.00071   2.481     0.797     2.221      0.757      yes     
map_lr1e-3    0.001   16        map     0        10          0.00657     0.00645   -0.00012  2.271     0.772     1.959      0.735      n